# 背景污染诊断：按平台抽样检查

从 Objaverse 四个平台各抽 5 个模型 → 体素化 → 可视化对比
目标：确认背景乱入是 Sketchfab 独有的问题还是普遍现象

In [ ]:
!pip install objaverse trimesh matplotlib --quiet

In [ ]:
import objaverse
import numpy as np
import matplotlib.pyplot as plt
import os, shutil, gc
from collections import defaultdict

SAMPLE_PER_SOURCE = 5
VOXEL_RES = 64

os.makedirs('/content/diag_models', exist_ok=True)
os.makedirs('/content/diag_voxels', exist_ok=True)

print('Loading annotations...')
annotations = objaverse.load_annotations()

# DEBUG: 查看 annotation 有哪些字段
first_uid = next(iter(annotations))
first_anno = annotations[first_uid]
print(f'\nAnnotation keys: {list(first_anno.keys())}')
for k, v in first_anno.items():
    print(f'  {k}: {str(v)[:200]}')
print()

# 按平台分组，每个平台挑有描述+有tag的模型
by_source = defaultdict(list)
category_kw = ['chair', 'table', 'desk', 'lamp', 'sofa', 'bench', 'stool', 'cabinet']

for uid, anno in annotations.items():
    # v1 annotations 没有 source/fileIdentifier，从 uri 字段推断
    uri = str(anno.get('uri', '')).lower()
    if 'sketchfab.com' in uri:
        source = 'sketchfab'
    elif 'github.com' in uri:
        source = 'github'
    elif 'thingiverse.com' in uri:
        source = 'thingiverse'
    elif '3d.si.edu' in uri:
        source = 'smithsonian'
    else:
        source = 'sketchfab'  # v1 API 只有 Sketchfab
    # 切换到 objaverse.xl 才能获取多平台模型
    if 'sketchfab.com' in fid: source = 'sketchfab'
    elif 'github.com' in fid: source = 'github'
    elif 'thingiverse.com' in fid: source = 'thingiverse'
    elif '3d.si.edu' in fid: source = 'smithsonian'
    name = str(anno.get('name', '')).lower()
    tags = ' '.join([str(t.get('name', '')).lower() for t in anno.get('tags', [])])
    desc = str(anno.get('description', ''))

    text = name + ' ' + tags
    if any(kw in text for kw in category_kw) and len(desc) > 30:
        if len(by_source[source]) < SAMPLE_PER_SOURCE:
            by_source[source].append(uid)

for src, uids in by_source.items():
    print(f'{src:20s}: {len(uids)} models')

# 收集所有 uid
all_uids = []
uid_to_source = {}
for src, uids in by_source.items():
    for uid in uids:
        all_uids.append(uid)
        uid_to_source[uid] = src

del annotations; gc.collect()
print(f'\nTotal: {len(all_uids)} models to inspect')

In [ ]:
# 下载模型
print(f'Downloading {len(all_uids)} models...')
downloaded = objaverse.load_objects(
    uids=all_uids,
    download_processes=4
)

# 保存 uid->source 映射（避免后续重新加载 annotations）
import json
with open('/content/uid_to_source.json', 'w') as f:
    json.dump(uid_to_source, f)

for uid, path in downloaded.items():
    with open('/content/uid_to_source.json', 'w') as f:
        json.dump(uid_to_source, f)
        print('Saved uid_to_source.json')
        if os.path.exists(path):
            shutil.copy(path, os.path.join('/content/diag_models', f'{uid}.glb'))

saved = os.listdir('/content/diag_models')
print(f'Downloaded {len(saved)} models')

In [ ]:
# 体素化函数（原始方式，不做过滤）
import trimesh

def sample_colors(mesh, points, face_idx):
    """从 mesh 采样颜色，PBR贴图 → PBR纯色 → 顶点色 → 灰色"""
    try:
        # 路径 1: PBR baseColorTexture (带贴图)
        if (hasattr(mesh.visual, 'material')
                and hasattr(mesh.visual.material, 'baseColorTexture')
                and mesh.visual.material.baseColorTexture is not None):
            texture = np.array(mesh.visual.material.baseColorTexture)
            uv = mesh.visual.uv
            if uv is not None and len(uv) > 0:
                face_uv = uv[mesh.faces[face_idx]]
                bary = trimesh.triangles.points_to_barycentric(
                    mesh.triangles[face_idx], points)
                sample_uv = (face_uv * bary[:, :, None]).sum(axis=1)
                H, W = texture.shape[:2]
                u, v = sample_uv[:, 0], 1.0 - sample_uv[:, 1]
                px = np.clip((u * (W - 1)).astype(int), 0, W - 1)
                py = np.clip((v * (H - 1)).astype(int), 0, H - 1)
                return texture[py, px][:, :3]
    except Exception:
        pass

    try:
        # 路径 2: PBR baseColorFactor (纯色，无贴图)
        if (hasattr(mesh.visual, 'material')
                and hasattr(mesh.visual.material, 'baseColorFactor')):
            c = np.array(mesh.visual.material.baseColorFactor[:3]) * 255
            return np.tile(c.astype(np.uint8), (len(points), 1))
    except Exception:
        pass

    try:
        # 路径 3: vertex_colors
        if hasattr(mesh.visual, 'vertex_colors') and mesh.visual.vertex_colors is not None:
            vc = mesh.visual.vertex_colors
            if vc.ndim == 2 and vc.shape[1] >= 3:
                from trimesh.visual.color import vertex_to_face_color
                fc = vertex_to_face_color(vc, mesh.faces)
                return fc[face_idx][:, :3]
    except Exception:
        pass

    # 路径 4: 灰色默认
    return np.tile([128, 128, 128], (len(points), 1))


def voxelize_raw(file_path, resolution=64):
    """原始体素化：不区分子 mesh，整体一起处理"""
    try:
        scene = trimesh.load(file_path)
        if isinstance(scene, trimesh.Scene):
            mesh = trimesh.util.concatenate(tuple(scene.geometry.values()))
        else:
            mesh = scene

        n_samples = 200000
        points, face_idx = trimesh.sample.sample_surface(mesh, n_samples)
        colors = sample_colors(mesh, points, face_idx)

        mins, maxs = points.min(axis=0), points.max(axis=0)
        points_norm = (points - mins) / (maxs - mins + 1e-8)
        coords = (points_norm * (resolution - 1)).astype(int)
        coords = np.clip(coords, 0, resolution - 1)

        voxel = np.zeros((resolution, resolution, resolution, 4), dtype=np.uint8)
        for c, color in zip(coords, colors):
            voxel[c[0], c[1], c[2], :3] = color
            voxel[c[0], c[1], c[2], 3] = 255

        return voxel
    except Exception as e:
        return None


# 体素化所有模型 —— 从保存的映射文件读取 source

# Read source mapping saved during download (no need to reload annotations)
import json
try:
    with open('/content/uid_to_source.json', 'r') as f:
        uid_to_source = json.load(f)
    print(f'Loaded source mapping: {len(uid_to_source)} UIDs')
except FileNotFoundError:
    print('ERROR: uid_to_source.json not found. Rerun the download cell!')
    uid_to_source = {}

results = {}

model_dir = '/content/diag_models'
for filename in os.listdir(model_dir):
    if not filename.endswith('.glb'):
        continue
    uid = filename.replace('.glb', '')
    file_path = os.path.join(model_dir, filename)

    source = uid_to_source.get(uid, 'unknown')
    model_name = uid[:20]

    # 统计子 mesh 详情
    submesh_names = []
    try:
        scene = trimesh.load(file_path, force='scene')
        if isinstance(scene, trimesh.Scene):
            submesh_names = [g.metadata.get('name', f'mesh_{i}')
                             if hasattr(g, 'metadata') else f'mesh_{i}'
                             for i, g in enumerate(scene.geometry.values())]
            submesh_count = len(submesh_names)
        else:
            submesh_count = 1
            submesh_names = ['main']
    except Exception:
        submesh_count = '?'
        submesh_names = ['?']

    voxel = voxelize_raw(file_path, VOXEL_RES)
    if voxel is not None:
        results[uid] = {
            'voxel': voxel,
            'source': source,
            'submesh_count': submesh_count,
            'submesh_names': submesh_names,
            'uid': uid,
            'name': model_name,
        }
        colored = 'color' if np.any(voxel[..., :3].sum(axis=-1) != 128*3*voxel[..., 3].astype(bool)) else 'gray'
        print(f'  [{source:<12s}] {model_name[:40]:<42s} {submesh_count} parts -> {colored}')
        if len(submesh_names) <= 10:
            print(f'             Parts: {submesh_names}')
    else:
        print(f'  [{source:<12s}] {model_name[:40]:<42s} FAILED')

print(f'\nVoxelized {len(results)} / {len(saved)} models successfully')

In [ ]:
# 按平台分组展示所有体素
from collections import defaultdict

print(f'DEBUG: results has {len(results)} models')
if len(results) == 0:
    print('ERROR: results is empty. Rerun the voxelization cell above.')
else:
    grouped = defaultdict(list)
    for uid, info in results.items():
        grouped[info['source']].append(info)

    print(f'DEBUG: Sources found: {dict((k, len(v)) for k, v in grouped.items())}')
    print()

    for source in sorted(grouped.keys()):
        items = grouped[source]
        n = len(items)
        cols = min(n, 5)
        rows = max(1, (n + cols - 1) // cols)

        fig = plt.figure(figsize=(4 * cols, 4 * rows))
        fig.suptitle(f'Source: {source.upper()} ({n} models)',
                     fontsize=14, fontweight='bold', y=0.98)

        for i, item in enumerate(items):
            voxel = item['voxel']
            occ = voxel[..., 3] > 0
            colors = voxel[..., :3].astype(float) / 255.0

            try:
                ax = fig.add_subplot(rows, cols, i + 1, projection='3d')
                ax.voxels(occ, facecolors=colors, edgecolor=None)
                sparsity = occ.mean() * 100
                ax.set_title(
                    f'{item["uid"][:12]}... | {item["submesh_count"]} sub | {sparsity:.1f}%',
                    fontsize=9
                )
                ax.axis('off')
            except Exception as e:
                print(f'  Render error for {item["uid"][:12]}: {e}')

        plt.tight_layout()
        path = f'/content/diag_voxels/{source}_samples.png'
        plt.savefig(path, dpi=100, bbox_inches='tight')
        plt.show()
        print(f'Saved: {path}\n')

In [ ]:
# 汇总统计
print('=' * 70)
print(f'{"Source":<20s} {"Models":<8s} {"Multi-part":<12s} {"Avg fill %":<12s} {"Notes"}')
print('-' * 70)

for source in sorted(grouped.keys()):
    items = grouped[source]
    if not items:
        continue

    multi_part = sum(1 for i in items if isinstance(i['submesh_count'], int) and i['submesh_count'] > 1)
    avg_fill = np.mean([i['voxel'][..., 3].mean() * 100 for i in items])

    # 简单的背景嫌疑判定：子 mesh > 3 且 体积填充 < 15%
    suspect = sum(
        1 for i in items
        if isinstance(i['submesh_count'], int)
        and i['submesh_count'] > 3
        and i['voxel'][..., 3].mean() < 0.15
    )

    notes = ""
    if suspect > 0:
        notes = f"{suspect} models suspect (many parts + low fill → possible background)"

    print(f'{source:<20s} {len(items):<8d} {multi_part:<12d} {avg_fill:<12.1f} {notes}')

print('=' * 70)
print('\n判定逻辑：子mesh数 > 3 + 体素填充率 < 15% = 嫌疑（可能有背景平面占了大量空间）')
print('但这只是初步信号，最终需要肉眼确认。')

In [ ]:
# 手动标记区域 —— 逐个检查你觉得可疑的模型
# 修改下面的 uid 来查看特定模型

INSPECT_UID = None  # 改成你想看的 uid，例如 'abc123...'

if INSPECT_UID and INSPECT_UID in results:
    item = results[INSPECT_UID]
    voxel = item['voxel']
    occ = voxel[..., 3] > 0
    colors = voxel[..., :3].astype(float) / 255.0

    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(projection='3d')
    ax.voxels(occ, facecolors=colors, edgecolor=None)
    ax.set_title(f"Source: {item['source']} | UID: {item['uid']} | Parts: {item['submesh_count']}")
    plt.show()

    # 同时显示原始模型信息
    anno = objaverse.load_annotations()
    if INSPECT_UID in anno:
        a = anno[INSPECT_UID]
        print(f"Name: {a.get('name', 'N/A')}")
        print(f"Description: {a.get('description', 'N/A')[:200]}")
        print(f"Tags: {[t.get('name') for t in a.get('tags', [])]}")
else:
    print("修改 INSPECT_UID 变量为你怀疑的模型 uid，重新运行此单元格")

In [ ]:
# 保存结果
import json

summary = {}
for source in sorted(grouped.keys()):
    items = grouped[source]
    summary[source] = {
        'count': len(items),
        'uids': [i['uid'] for i in items],
        'submesh_counts': [i['submesh_count'] for i in items],
        'fill_ratios': [float(i['voxel'][..., 3].mean()) for i in items],
    }

with open('/content/diag_voxels/diagnosis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Diagnosis saved to /content/diag_voxels/diagnosis_summary.json')
print('Done. 查看每张按平台的分组图，确认哪个平台有背景污染问题。')